In [ ]:
# Written by Arijit Bhattacharya, Ashoka University and University of Oxford

import numpy as np, nibabel as nib, matplotlib.pyplot as plt
from fsl_mrs.utils.misc import FIDToSpec
from fsl_mrs.utils import mrs_io

DATA = "/path/to/raw/data.nii.gz"
FIT  = "/path/to/fitting/fit.nii.gz"
PPMLIM, REF = (0.2, 4.2), 4.65
RESID_RANGE = "window"      # "window" (truncated) or "full"

# Load & transform
load = lambda p: np.asanyarray(nib.load(p).dataobj).reshape(-1).astype(np.complex128)
da, fa = load(DATA), load(FIT)
assert da.size == fa.size, "data and fit have different sizes"
n = da.size

obj = mrs_io.read_FID(DATA)
bw = float(np.atleast_1d(obj.bandwidth).ravel()[0])
sf = float(np.atleast_1d(obj.spectrometer_frequency).ravel()[0])
if sf > 1e5: sf /= 1e6

S_d = np.asarray(FIDToSpec(da)).astype(np.complex128)
S_f = np.asarray(FIDToSpec(fa)).astype(np.complex128)

bad = [np.isnan(S_d).any(), np.isnan(S_f).any(), np.isinf(S_d).any(), np.isinf(S_f).any()]
print(f"dtypes: {S_d.dtype}, {S_f.dtype} | NaN/Inf present: {any(bad)}")
assert not any(bad), "NaN or Inf in spectra"

# ppm axis: N samples -> N bins -> N ppm values (derived, not measured)
#   freq[k] = -bw/2 + k*bw/N ;  ppm[k] = -freq[k]/sf + REF
freq = np.fft.fftshift(np.fft.fftfreq(n, d=1.0 / bw))
ppm  = -freq / sf + REF
lo, hi = PPMLIM

# TRUNCATION: ppm limits -> bin indices first:last
idx = np.where((ppm >= lo) & (ppm <= hi))[0]
first, last = int(idx[0]), int(idx[-1]) + 1
w = slice(first, last)                       # truncated spectrum = S[first:last]
N_win = last - first
print(f"truncation [{first}:{last}]  N = {N_win}  "
      f"covers {ppm[w].min():.3f}..{ppm[w].max():.3f} ppm  (dppm = {bw/n/sf:.5f})\n")

rss = lambda r, m=w: float(np.sum(np.abs(r[m])**2))


# Plot helper: top-left full, top-right truncated, bottom = residual only

LANDMARKS = (2.01, 3.03, 3.22)   # NAA, tCr, tCho

def step_plot(title, d, f, r, ylab, signed):
    fig = plt.figure(figsize=(14, 9))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.3, 1])
    a_full = fig.add_subplot(gs[0, 0])
    a_win  = fig.add_subplot(gs[0, 1])
    a_res  = fig.add_subplot(gs[1, :])

    a_full.plot(ppm, d, lw=0.8, color='C0', label='data')
    a_full.plot(ppm, f, lw=0.8, color='C1', label='fit', alpha=0.85)
    a_full.axvspan(lo, hi, color='g', alpha=0.12)
    a_full.set_xlim(ppm.max(), ppm.min())
    a_full.set_title("Full spectrum")

    a_win.plot(ppm[w], d[w], lw=1.0, color='C0', label='data')
    a_win.plot(ppm[w], f[w], lw=1.0, color='C1', label='fit')
    a_win.set_xlim(hi, lo)
    a_win.set_title(f"Truncated [{first}:{last}] -> [{lo}, {hi}] ppm")

    if RESID_RANGE == "window":
        a_res.plot(ppm[w], r[w], lw=1.0, color='C3')
        a_res.set_xlim(hi, lo)
        a_res.set_title(f"Residual only, truncated [{lo}, {hi}] ppm")
    else:
        a_res.plot(ppm, r, lw=0.8, color='C3')
        a_res.axvspan(lo, hi, color='g', alpha=0.12)
        a_res.set_xlim(ppm.max(), ppm.min())
        a_res.set_title("Residual only, full spectrum")

    for a, yl in ((a_full, ylab), (a_win, ylab), (a_res, "residual")):
        a.set_xlabel("ppm"); a.set_ylabel(yl)
        if signed: a.axhline(0, color='grey', lw=0.5)
        for p in LANDMARKS:
            xa, xb = sorted(a.get_xlim())
            if xa <= p <= xb:
                a.axvline(p, color='grey', lw=0.4, ls=':', alpha=0.6)
    a_full.legend(fontsize=8); a_win.legend(fontsize=8)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

# STEP 1: |data − fit|
res_mag = np.abs(S_d - S_f)
step_plot("Step 1: magnitude. Bottom = |data − fit|",
          np.abs(S_d), np.abs(S_f), res_mag, "magnitude", signed=False)
rss1 = rss(res_mag)
print("STEP 1  |data − fit|")
print(f"  RSS truncated = {rss1:.4e}   RSS full = {rss(res_mag, slice(None)):.4e}\n")

# STEP 2: Re(data) − Re(fit)      # Here re= real component of the data
res_re = S_d.real - S_f.real
step_plot("Step 2: real part. Bottom = Re(data) − Re(fit)",
          S_d.real, S_f.real, res_re, "Re", signed=True)
rss2 = rss(res_re)
print("STEP 2  Re(data) − Re(fit)")
print(f"  RSS truncated = {rss2:.4e}   RSS full = {rss(res_re, slice(None)):.4e}\n")

# STEP 3: Im(data) − Im(fit)      # Here img= imaginary component of the data
res_im = S_d.imag - S_f.imag
step_plot("Step 3: imaginary part. Bottom = Im(data) − Im(fit)",
          S_d.imag, S_f.imag, res_im, "Im", signed=True)
rss3 = rss(res_im)
print("STEP 3  Im(data) − Im(fit)")
print(f"  RSS truncated = {rss3:.4e}   RSS full = {rss(res_im, slice(None)):.4e}\n")

# STEP 4: (Re res) + (Im res) = (Re d + Im d) − (Re f + Im f)

sum_d   = S_d.real + S_d.imag
sum_f   = S_f.real + S_f.imag
res_sum = res_re + res_im

scale = max(np.abs(S_d).max(), np.abs(S_f).max())
err   = float(np.max(np.abs(res_sum - (sum_d - sum_f))))
print(f"Step 4 consistency: max |(Re_res+Im_res) − direct| = {err:.3e} "
      f"({err/scale:.1e} of data peak) -> {'OK' if err <= 1e-9 * scale else 'CHECK'}")

step_plot("Step 4: Re+Im. Bottom = Re_res + Im_res = (Re d+Im d) − (Re f+Im f)",
          sum_d, sum_f, res_sum, "Re + Im", signed=True)
rss4  = rss(res_sum)
cross = 2 * float(np.sum(res_re[w] * res_im[w]))
print("STEP 4  Re_res + Im_res")
print(f"  RSS truncated = {rss4:.4e}   RSS full = {rss(res_sum, slice(None)):.4e}")
print(f"  RSS_re + RSS_im = {rss2 + rss3:.4e}")
print(f"  difference = {rss4 - (rss2 + rss3):+.4e}  (cross term 2·ΣRe·Im = {cross:+.4e})\n")

# Summary

n_eff = 2 * N_win
print("=" * 60)
print(f"{'Step':<32}{'RSS (truncated)':>16}")
print("-" * 60)
for name, v in (("1  |data − fit|", rss1),
                ("2  Re residual", rss2),
                ("3  Im residual", rss3),
                ("   (2 + 3, sum of squares)", rss2 + rss3),
                ("4  Re_res + Im_res (signed)", rss4)):
    print(f"{name:<32}{v:>16.4e}")
print("-" * 60)
print(f"Step 1 == RSS_re + RSS_im : {np.isclose(rss1, rss2 + rss3)}")
print(f"N complex = {N_win},  n_eff = {n_eff}")
print(f"-2·logL (Step 1) = {n_eff * np.log(rss1 / n_eff):.4e}")